# AZT1D: estimar a seta de tendência glic?mica

Este notebook reúne preparação, baselines, treino dos modelos de 5 e 15 minutos,
seleção do horizonte e avaliação final em um ?nico fluxo. A saída ? uma taxa
contínua de variação e sua representa??o como seta, não uma classificação em
três categorias.

Execute as células em ordem. Para uma verificação rápida, mantenha `QUICK = True`;
para o experimento completo, use outro `RUN_NAME` e `QUICK = False`. Fora do Colab,
o notebook localiza automaticamente `AZT1D 2025/CGM Records` e salva os resultados
em `results/azt1d_seta`; você também pode definir `AZT1D_LOCAL_DATA` e
`AZT1D_LOCAL_PROJECT`. No Colab, envie o ZIP do AZT1D ou use a pasta no Drive.

A seleção compara os horizontes na validação antes de consultar o teste. O modo
rápido serve para conferir o fluxo e não constitui resultado final do estudo.


In [ ]:
# @title 1. Ambiente, Google Drive e identificação da execução
import os, sys, subprocess, importlib.util, importlib.metadata
from pathlib import Path

# AZT1D_LOCAL_PROJECT permite execução local automatizada fora do Colab.
IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
USE_DRIVE = IN_COLAB and not bool(os.environ.get("AZT1D_LOCAL_PROJECT"))
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "AZT1D 2025" / "CGM Records").is_dir()), Path.cwd())
PROJECT_PATH = "/content/drive/MyDrive/TCC_AZT1D_Seta" # @param {type:"string"}
RUN_NAME = os.environ.get("AZT1D_RUN_NAME", "quick_v1") # @param {type:"string"}

requirements = {"numpy": "numpy>=1.26,<3", "pandas": "pandas>=2.0,<3",
                "matplotlib": "matplotlib>=3.8,<4"}
NEED_TORCH = True
if NEED_TORCH:
    requirements["torch"] = "torch>=2.2,<3"
missing = [spec for name, spec in requirements.items()
           if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
if int(importlib.metadata.version("pandas").split(".")[0]) < 2:
    raise RuntimeError("Requer pandas >=2. Atualize o runtime ou instale pandas>=2 e reinicie.")

if USE_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError("Fora do Colab: defina AZT1D_LOCAL_PROJECT.") from exc
    drive.mount("/content/drive")
    PROJECT_DIR = Path(PROJECT_PATH)
else:
    PROJECT_DIR = Path(os.environ.get("AZT1D_LOCAL_PROJECT", REPO_ROOT / "results" / "azt1d_seta"))
import re
if not re.fullmatch(r"[A-Za-z0-9_-]+", RUN_NAME):
    raise ValueError("RUN_NAME: use letras, números, hífen e sublinhado.")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = PROJECT_DIR / "runs" / RUN_NAME
CORE_PATH = PROJECT_DIR / "azt1d_core.py"
print("Projeto:", PROJECT_DIR)
print("Execução:", RUN_DIR)

In [ ]:
# @title 2. Salvar funções compartilhadas (execute; esta célula pode ficar recolhida)
CORE_SOURCE = '"""Experimentos reprodutíveis de taxa futura de CGM: AZT1D, 5 e 15 min."""\nfrom pathlib import Path\nimport hashlib\nimport json\nimport platform\nimport random\nimport re\nimport time\nimport zipfile\n\nimport numpy as np\nimport pandas as pd\n\nVERSION = "1.0.0"\nHORIZONS = (5, 15)\nMODELS = ("persistencia", "ultima_variacao", "linear_15min", "lstm")\nSTEP_NS = 5 * 60 * 10**9\n\n\ndef clean_json(value):\n    if isinstance(value, dict):\n        return {str(k): clean_json(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [clean_json(v) for v in value]\n    if isinstance(value, np.ndarray):\n        return clean_json(value.tolist())\n    if isinstance(value, (np.integer, np.floating)):\n        value = value.item()\n    if isinstance(value, float) and not np.isfinite(value):\n        return None\n    if isinstance(value, Path):\n        return str(value)\n    return value\n\n\ndef save_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(clean_json(value), ensure_ascii=False,\n                               indent=2, allow_nan=False), encoding="utf-8")\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef file_sha(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for block in iter(lambda: f.read(1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef safe_extract_zip(archive, destination):\n    """Extrai somente dentro da pasta indicada; nunca executa conteúdo do ZIP."""\n    destination = Path(destination).resolve()\n    destination.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(archive) as z:\n        for info in z.infolist():\n            target = (destination / info.filename).resolve()\n            if not target.is_relative_to(destination):\n                raise ValueError("ZIP contém caminho fora da pasta de destino.")\n            if (info.external_attr >> 16) & 0o170000 == 0o120000:\n                raise ValueError("ZIP com link simbólico: extraia manualmente.")\n        z.extractall(destination)\n    return destination\n\n\ndef subject_number(path):\n    for piece in reversed(Path(path).parts):\n        match = re.search(r"subject[\\s_-]*(\\d+)", piece, re.I)\n        if match:\n            return int(match.group(1))\n    return None\n\n\ndef discover_csvs(data_root):\n    root = Path(data_root)\n    if not root.exists():\n        raise FileNotFoundError(f"Pasta do AZT1D não encontrada: {root}")\n    files = sorted(p for p in root.rglob("*")\n                   if p.suffix.lower() == ".csv" and "__MACOSX" not in p.parts\n                   and subject_number(p) is not None)\n    if not files:\n        raise ValueError("Nenhum CSV em pastas Subject N. Confira DATA_DIR ou o ZIP.")\n    return files\n\n\ndef read_csv_flexible(path):\n    for encoding in ("utf-8-sig", "cp1252"):\n        try:\n            df = pd.read_csv(path, sep=None, engine="python", dtype=str,\n                             encoding=encoding)\n            df.columns = [str(c).strip().lstrip("\\ufeff") for c in df.columns]\n            return df\n        except UnicodeDecodeError:\n            continue\n    raise ValueError(f"Não foi possível ler a codificação de {path}")\n\n\ndef clean_subject(frames, subject, dayfirst=False, date_format="mixed"):\n    """Uma origem de grade nunca recebe uma leitura com timestamp posterior."""\n    raw = pd.concat(frames, ignore_index=True)\n    n_raw = len(raw)\n    raw = raw.drop_duplicates()\n    n_duplicates = n_raw - len(raw)\n    date_examples = raw["EventDateTime"].dropna().head(3).tolist()\n    ts = pd.to_datetime(raw["EventDateTime"], format=date_format,\n                        dayfirst=dayfirst, errors="coerce")\n    if getattr(ts.dt, "tz", None) is not None:\n        raise ValueError("Timestamps com fuso: normalize explicitamente antes de usar.")\n    values = pd.to_numeric(raw["CGM"].str.replace(",", ".", regex=False),\n                           errors="coerce")\n    invalid_g = ~np.isfinite(values) | (values <= 0)\n    values = values.mask(invalid_g)\n    valid = pd.DataFrame({"timestamp_raw": ts, "glucose": values}).dropna(\n        subset=["timestamp_raw"]).sort_values("timestamp_raw")\n    if valid.empty or valid["glucose"].notna().sum() < 15:\n        raise ValueError(f"Subject {subject}: menos de 15 leituras válidas.")\n    if valid["glucose"].median() < 30:\n        raise ValueError(f"Subject {subject}: verifique unidade; esperado mg/dL.")\n    valid["grid_time"] = valid["timestamp_raw"].dt.ceil("5min")\n    grouped = valid.groupby("grid_time")["glucose"]\n    g = grouped.first()\n    distinct = grouped.nunique()\n    g[distinct != 1] = np.nan\n    raw_time = valid.dropna(subset=["glucose"]).groupby("grid_time")[\n        "timestamp_raw"].max()\n    index = pd.date_range(g.index.min(), g.index.max(), freq="5min")\n    if len(index) > 200_000:\n        raise ValueError("Período inesperadamente longo: confira o formato das datas.")\n    grid = pd.DataFrame({"glucose": g, "timestamp_raw": raw_time}).reindex(index)\n    grid.index.name = "grid_time"\n    grid.loc[grid["glucose"].isna(), "timestamp_raw"] = pd.NaT\n    good = grid["timestamp_raw"].notna()\n    assert (grid.loc[good, "timestamp_raw"].to_numpy() <=\n            grid.index[good].to_numpy()).all()\n    age = (grid.index[good].to_numpy() -\n           grid.loc[good, "timestamp_raw"].to_numpy()) / np.timedelta64(1, "m")\n    audit = {\n        "subject": subject, "raw_rows": n_raw, "duplicates_removed": n_duplicates,\n        "invalid_timestamps": int(ts.isna().sum()),\n        "invalid_glucose": int(invalid_g.sum()), "grid_bins": len(grid),\n        "conflict_bins": int((distinct > 1).sum()),\n        "missing_or_conflict_bins": int(grid.glucose.isna().sum()),\n        "valid_bins": int(grid.glucose.notna().sum()),\n        "start": str(index.min()), "end": str(index.max()),\n        "raw_date_examples": " | ".join(map(str, date_examples)),\n        "reading_age_max_min": float(np.max(age)),\n        "reading_age_median_min": float(np.median(age)),\n        "values_40_or_400": int(grid.glucose.isin([40, 400]).sum()),\n    }\n    return grid, audit\n\n\ndef windows_from_grid(grid, subject):\n    """Divide primeiro, depois constrói suporte completo de 12 + 3 pontos."""\n    n = len(grid)\n    c1, c2 = int(n * .60), int(n * .80)\n    bounds = [(0, c1), (c1, c2), (c2, n)]\n    pieces, split_rows = {}, []\n    for split, (lo, hi) in zip(("train", "val", "test"), bounds):\n        part = grid.iloc[lo:hi]\n        g = part.glucose.to_numpy(dtype=np.float32)\n        if len(g) < 15:\n            continue\n        windows = np.lib.stride_tricks.sliding_window_view(g, 15)\n        starts = np.flatnonzero(np.isfinite(windows).all(axis=1))\n        if len(starts) == 0:\n            continue\n        w = windows[starts]\n        i = starts + 11\n        times = part.index.to_numpy(dtype="datetime64[ns]").astype(np.int64)\n        raw = part.timestamp_raw.to_numpy(dtype="datetime64[ns]").astype(np.int64)\n        origins = times[i]\n        targets = times[i[:, None] + np.array([1, 3])]\n        future = w[:, [12, 14]].copy()\n        X = w[:, :12, None].copy()\n        y = (future - X[:, -1, 0, None]) / np.array([5., 15.], np.float32)\n        pieces[split] = {\n            "X": X, "y": y.astype(np.float32), "future": future,\n            "subject": np.full(len(i), subject, dtype=np.int32),\n            "origin_ns": origins, "target_ns": targets,\n            "input_start_ns": times[starts], "raw_origin_ns": raw[i],\n            "raw_target_ns": raw[i[:, None] + np.array([1, 3])],\n        }\n        assert np.all(targets - origins[:, None] ==\n                      np.array([1, 3]) * STEP_NS)\n        assert np.all(times[starts] >= times[0]) and np.all(targets <= times[-1])\n        np.testing.assert_allclose(X[:, -1, 0, None] + y * [5, 15],\n                                   future, atol=3e-5)\n        split_rows.append({\n            "subject": subject, "split": split, "start": str(part.index[0]),\n            "end": str(part.index[-1]), "windows_before_sampling": len(i),\n        })\n    return pieces, split_rows\n\n\ndef prepare_dataset(data_root, run_dir, quick=True, dayfirst=False,\n                    date_format="mixed", allow_mixed_alias=False,\n                    seed=42, min_windows=1):\n    started = time.perf_counter()\n    run_dir, data_root = Path(run_dir), Path(data_root)\n    files = discover_csvs(data_root)\n    source = [{"path": str(p.relative_to(data_root)), "sha256": file_sha(p),\n               "bytes": p.stat().st_size} for p in files]\n    cfg = dict(quick=bool(quick), dayfirst=bool(dayfirst), date_format=date_format,\n               allow_mixed_alias=bool(allow_mixed_alias), seed=int(seed),\n               min_windows=int(min_windows), version=VERSION,\n               core_sha256=file_sha(__file__))\n    fingerprint = hashlib.sha256(json.dumps(\n        [source, cfg], sort_keys=True).encode()).hexdigest()\n    manifest_path = run_dir / "manifest.json"\n    if manifest_path.exists():\n        old = read_json(manifest_path)\n        if old["fingerprint"] != fingerprint:\n            raise ValueError("RUN_NAME já existe com dados/configuração diferentes. "\n                             "Use um novo RUN_NAME; não misture experimentos.")\n        if not all((run_dir / "prepared" / f"{s}.npz").exists()\n                   for s in ("train", "val", "test")):\n            raise ValueError("Cache incompleto. Use um novo RUN_NAME.")\n        return old\n    if run_dir.exists() and any(run_dir.iterdir()):\n        raise ValueError("Pasta de execução incompleta ou ocupada. Use novo RUN_NAME.")\n    frames, file_audit, excluded = {}, [], []\n    for p in files:\n        df = read_csv_flexible(p)\n        sid = subject_number(p)\n        names = {c.lower(): c for c in df.columns}\n        date_col = names.get("eventdatetime")\n        glucose_col = names.get("cgm")\n        mixed = names.get("readings (cgm / bgm)")\n        if not date_col or not (glucose_col or mixed):\n            file_audit.append({"file": str(p.relative_to(data_root)),\n                               "subject": sid, "status": "colunas_ausentes",\n                               "columns": " | ".join(df.columns)})\n            continue\n        if not glucose_col and mixed and not allow_mixed_alias:\n            excluded.append({"subject": sid, "reason": "alias_CGM_BGM_nao_confirmado"})\n            file_audit.append({"file": str(p.relative_to(data_root)),\n                               "subject": sid, "status": "alias_excluido"})\n            continue\n        df = df.rename(columns={date_col: "EventDateTime",\n                                 glucose_col or mixed: "CGM"})\n        frames.setdefault(sid, []).append(df)\n        file_audit.append({"file": str(p.relative_to(data_root)),\n                           "subject": sid, "status": "lido", "rows": len(df)})\n    if not frames:\n        raise ValueError("Não há CSVs elegíveis com EventDateTime e CGM.")\n    eligible, audit, splits, collections = [], [], [], []\n    for sid in sorted(frames):\n        grid, row = clean_subject(frames[sid], sid, dayfirst, date_format)\n        audit.append(row)\n        pieces, split_rows = windows_from_grid(grid, sid)\n        if not all(s in pieces and len(pieces[s]["X"]) >= min_windows\n                   for s in ("train", "val", "test")):\n            excluded.append({"subject": sid, "reason": "sem_janelas_nas_3_particoes"})\n            continue\n        if quick and len(eligible) >= 3:\n            continue\n        eligible.append(sid)\n        collections.append(pieces)\n        splits.extend(split_rows)\n    if not eligible:\n        raise ValueError("Nenhum participante tem janelas nas três partições.")\n    prepared = run_dir / "prepared"\n    prepared.mkdir(parents=True, exist_ok=True)\n    counts, timing_rows, artifact_hashes = {}, [], {}\n    for split in ("train", "val", "test"):\n        keys = collections[0][split].keys()\n        arr = {k: np.concatenate([c[split][k] for c in collections])\n               for k in keys}\n        before = len(arr["X"])\n        if quick and split == "train" and before > 20_000:\n            keep = np.sort(np.random.default_rng(seed).choice(\n                before, 20_000, replace=False))\n            arr = {k: v[keep] for k, v in arr.items()}\n        path = prepared / f"{split}.npz"\n        np.savez_compressed(path, **arr)\n        artifact_hashes[split] = file_sha(path)\n        counts[split] = {"before_sampling": before, "used": len(arr["X"])}\n        for sid in eligible:\n            mask = arr["subject"] == sid\n            for j, h in enumerate(HORIZONS):\n                intervals = (arr["raw_target_ns"][mask, j] -\n                             arr["raw_origin_ns"][mask]) / (60 * 1e9)\n                future_after_origin = (arr["raw_target_ns"][mask, j] -\n                                       arr["origin_ns"][mask]) / (60 * 1e9)\n                timing_rows.append({\n                    "split": split, "subject": sid, "horizon_min": h,\n                    "n": len(intervals),\n                    "raw_spacing_min": float(intervals.min()),\n                    "raw_spacing_median": float(np.median(intervals)),\n                    "raw_spacing_max": float(intervals.max()),\n                    "raw_spacing_p95_abs_deviation": float(\n                        np.quantile(np.abs(intervals - h), .95)),\n                    "target_after_grid_origin_min": float(future_after_origin.min()),\n                })\n    pd.DataFrame(audit).to_csv(run_dir / "auditoria_participantes.csv", index=False)\n    pd.DataFrame(file_audit).to_csv(run_dir / "auditoria_arquivos.csv", index=False)\n    pd.DataFrame(splits).to_csv(run_dir / "divisoes_temporais.csv", index=False)\n    pd.DataFrame(timing_rows).to_csv(run_dir / "auditoria_horizontes_reais.csv", index=False)\n    manifest = {\n        "fingerprint": fingerprint, "config": cfg, "source_files": source,\n        "participants": eligible, "excluded": excluded, "counts": counts,\n        "prepared_sha256": artifact_hashes,\n        "preparation_seconds": time.perf_counter() - started,\n        "dataset_doi": "10.17632/gk9m674wcx.1",\n        "created_utc": pd.Timestamp.now(tz="UTC").isoformat(),\n    }\n    save_json(manifest_path, manifest)\n    return manifest\n\n\ndef load_run(run_dir):\n    run_dir = Path(run_dir)\n    if not (run_dir / "manifest.json").exists():\n        raise FileNotFoundError("Execute primeiro o notebook 01 com este RUN_NAME.")\n    m = read_json(run_dir / "manifest.json")\n    if m["config"]["core_sha256"] != file_sha(__file__):\n        raise ValueError("Código comum diferente daquele usado na preparação.")\n    return m\n\n\ndef load_split(run_dir, split):\n    m = load_run(run_dir)\n    if split not in ("train", "val", "test"):\n        raise ValueError(split)\n    path = Path(run_dir) / "prepared" / f"{split}.npz"\n    if file_sha(path) != m["prepared_sha256"][split]:\n        raise ValueError("Cache de dados mudou. Use uma execução nova.")\n    with np.load(path, allow_pickle=False) as a:\n        return {k: a[k] for k in a.files}\n\n\ndef baseline_rates(X):\n    g = X[:, :, 0].astype(np.float64)\n    times = np.array([-15., -10., -5., 0.])\n    centered = times - times.mean()\n    return {\n        "persistencia": np.zeros(len(g)),\n        "ultima_variacao": (g[:, -1] - g[:, -2]) / 5,\n        "linear_15min": (g[:, -4:] @ centered) / (centered @ centered),\n    }\n\n\ndef arrow_values(rate):\n    q = np.asarray(rate, dtype=float)  # dividido por 1 mg/dL/min\n    norm = np.hypot(1, q)\n    return np.degrees(np.arctan(q)), 1 / norm, q / norm\n\n\ndef prediction_frame(arr, horizon, model, rate, split):\n    j = HORIZONS.index(horizon)\n    rate = np.asarray(rate, dtype=float).reshape(-1)\n    if len(rate) != len(arr["X"]) or not np.isfinite(rate).all():\n        raise ValueError("Previsões ausentes, não finitas ou desalinhadas.")\n    angle, vx, vy = arrow_values(rate)\n    current = arr["X"][:, -1, 0]\n    return pd.DataFrame({\n        "subject": arr["subject"], "split": split, "horizon_min": horizon,\n        "model": model, "origin_ns": arr["origin_ns"],\n        "origin": pd.to_datetime(arr["origin_ns"]),\n        "target_time": pd.to_datetime(arr["target_ns"][:, j]),\n        "glucose_current": current, "glucose_true": arr["future"][:, j],\n        "rate_true": arr["y"][:, j], "rate_pred": rate,\n        "delta_pred": horizon * rate,\n        "glucose_pred": current + horizon * rate,\n        "angle_deg": angle, "vector_x": vx, "vector_y": vy,\n    })\n\n\ndef metrics_table(predictions, rapid_threshold=1.0):\n    result = []\n    groups = predictions.groupby(["split", "horizon_min", "model"], sort=True)\n    for (split, horizon, model), df in groups:\n        subsets = {"all": df, "rapid": df[df.rate_true.abs() >= rapid_threshold]}\n        for subset, data in subsets.items():\n            subjects = sorted(df.subject.unique())\n            subject_rows = []\n            for scope, sid, part in (\n                [("pooled", "ALL", data)] +\n                [("subject", str(s), data[data.subject == s]) for s in subjects]\n            ):\n                n = len(part)\n                err = part.rate_pred.to_numpy() - part.rate_true.to_numpy()\n                ge = part.glucose_pred.to_numpy() - part.glucose_true.to_numpy()\n                prod = part.rate_true.to_numpy() * part.rate_pred.to_numpy()\n                nonzero = part.rate_true.to_numpy() != 0\n                base = {\n                    "split": split, "horizon_min": horizon, "model": model,\n                    "subset": subset, "scope": scope, "subject": sid, "n": n,\n                    "participants_evaluable": int(n > 0) if scope == "subject"\n                        else int(part.subject.nunique()),\n                    "mae_rate": float(np.mean(np.abs(err))) if n else np.nan,\n                    "rmse_rate": float(np.sqrt(np.mean(err**2))) if n else np.nan,\n                    "bias_rate": float(np.mean(err)) if n else np.nan,\n                    "mae_glucose": float(np.mean(np.abs(ge))) if n else np.nan,\n                    "rmse_glucose": float(np.sqrt(np.mean(ge**2))) if n else np.nan,\n                    "direction_n": int(nonzero.sum()),\n                    "sign_correct": float(np.mean(prod[nonzero] > 0))\n                        if nonzero.any() else np.nan,\n                    "sign_opposite": float(np.mean(prod[nonzero] < 0))\n                        if nonzero.any() else np.nan,\n                    "pred_exact_zero": float(np.mean(\n                        part.rate_pred.to_numpy()[nonzero] == 0))\n                        if nonzero.any() else np.nan,\n                }\n                result.append(base)\n                if scope == "subject":\n                    subject_rows.append(base)\n            tab = pd.DataFrame(subject_rows)\n            macro = dict(split=split, horizon_min=horizon, model=model,\n                         subset=subset, scope="macro", subject="MEAN",\n                         n=int(tab.n.sum()),\n                         participants_evaluable=int((tab.n > 0).sum()),\n                         direction_n=int(tab.direction_n.sum()))\n            for key in ("mae_rate", "rmse_rate", "bias_rate", "mae_glucose",\n                        "rmse_glucose", "sign_correct", "sign_opposite",\n                        "pred_exact_zero"):\n                macro[key] = float(tab[key].mean())\n            result.append(macro)\n    return pd.DataFrame(result)\n\n\ndef baseline_validation(run_dir):\n    run_dir = Path(run_dir)\n    arr = load_split(run_dir, "val")\n    frames = [prediction_frame(arr, h, name, rate, "val")\n              for h in HORIZONS for name, rate in baseline_rates(arr["X"]).items()]\n    predictions = pd.concat(frames, ignore_index=True)\n    metrics = metrics_table(predictions)\n    out = run_dir / "baselines"\n    out.mkdir(exist_ok=True)\n    predictions.to_csv(out / "previsoes_validacao.csv.gz", index=False)\n    metrics.to_csv(out / "metricas_validacao.csv", index=False)\n    return metrics\n\n\ndef make_model(hidden=32):\n    import torch\n    class RateLSTM(torch.nn.Module):\n        def __init__(self):\n            super().__init__()\n            self.lstm = torch.nn.LSTM(1, hidden, batch_first=True)\n            self.head = torch.nn.Linear(hidden, 1)\n        def forward(self, x):\n            states, _ = self.lstm(x)  # estado reiniciado por janela\n            return self.head(states[:, -1, :]).squeeze(-1)\n    return RateLSTM()\n\n\ndef train_horizon(run_dir, horizon, seed=42):\n    import torch\n    from torch.utils.data import DataLoader, TensorDataset\n    run_dir = Path(run_dir)\n    m = load_run(run_dir)\n    if horizon not in HORIZONS:\n        raise ValueError("Horizontes permitidos: 5 e 15.")\n    out = run_dir / f"lstm_h{horizon}"\n    if (out / "completed.json").exists():\n        saved = read_json(out / "completed.json")\n        if saved["fingerprint"] != m["fingerprint"] or saved["seed"] != seed:\n            raise ValueError("Modelo existente incompatível; crie novo RUN_NAME.")\n        print(f"Reutilizando treinamento concluído: {out.name}")\n        return pd.read_csv(out / "metricas_validacao.csv")\n    if (run_dir / "selecao_horizonte.json").exists():\n        raise ValueError("Seleção já congelada: novo treinamento exige novo RUN_NAME.")\n    out.mkdir(parents=True, exist_ok=True)\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    torch.set_num_threads(min(4, torch.get_num_threads()))\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    train, val = load_split(run_dir, "train"), load_split(run_dir, "val")\n    j = HORIZONS.index(horizon)\n    scales = {\n        "x_mean": float(train["X"].mean()), "x_std": float(train["X"].std()),\n        "y_mean": float(train["y"][:, j].mean()),\n        "y_std": float(train["y"][:, j].std()),\n    }\n    scales["x_std"] = max(scales["x_std"], 1e-6)\n    scales["y_std"] = max(scales["y_std"], 1e-6)\n    def tensors(a):\n        x = (a["X"] - scales["x_mean"]) / scales["x_std"]\n        y = (a["y"][:, j] - scales["y_mean"]) / scales["y_std"]\n        return TensorDataset(torch.from_numpy(x.astype(np.float32)),\n                             torch.from_numpy(y.astype(np.float32)))\n    generator = torch.Generator().manual_seed(seed)\n    train_loader = DataLoader(tensors(train), batch_size=64, shuffle=True,\n                              generator=generator, num_workers=0)\n    val_loader = DataLoader(tensors(val), batch_size=1024, shuffle=False, num_workers=0)\n    model = make_model().to(device)\n    optimizer = torch.optim.Adam(model.parameters(), lr=.001)\n    loss_fn = torch.nn.MSELoss()\n    epochs = 3 if m["config"]["quick"] else 15\n    config = dict(horizon_min=horizon, seed=seed, hidden=32, batch_size=64,\n                  lr=.001, epochs=epochs, patience=3, device=str(device),\n                  fingerprint=m["fingerprint"], scales=scales, version=VERSION)\n    save_json(out / "config.json", config)\n    save_json(out / "ambiente.json", {\n        "python": platform.python_version(), "torch": torch.__version__,\n        "numpy": np.__version__, "pandas": pd.__version__,\n        "cuda": torch.version.cuda,\n        "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,\n    })\n    history, best, stale = [], float("inf"), 0\n    started = time.perf_counter()\n    for epoch in range(1, epochs + 1):\n        model.train(); total = 0.\n        for xb, yb in train_loader:\n            xb, yb = xb.to(device), yb.to(device)\n            optimizer.zero_grad(set_to_none=True)\n            loss = loss_fn(model(xb), yb)\n            if not torch.isfinite(loss):\n                raise RuntimeError("Perda não finita. Confira os dados e escalas.")\n            loss.backward()\n            optimizer.step()\n            total += float(loss.detach().cpu()) * len(xb)\n        model.eval(); val_total = 0.\n        with torch.no_grad():\n            for xb, yb in val_loader:\n                val_total += float(loss_fn(model(xb.to(device)),\n                                          yb.to(device)).cpu()) * len(xb)\n        train_loss = total / len(train["X"])\n        val_loss = val_total / len(val["X"])\n        history.append(dict(epoch=epoch, train_mse_scaled=train_loss,\n                            val_mse_scaled=val_loss,\n                            elapsed_seconds=time.perf_counter() - started))\n        pd.DataFrame(history).to_csv(out / "historico.csv", index=False)\n        if val_loss < best:\n            best, stale = val_loss, 0\n            torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()},\n                       out / "best_weights.pt")\n        else:\n            stale += 1\n        print(f"h={horizon} | época {epoch}/{epochs} | "\n              f"treino={train_loss:.5f} | validação={val_loss:.5f}")\n        if stale >= 3:\n            break\n    elapsed = time.perf_counter() - started\n    rates, inference_s = predict_horizon(run_dir, horizon, val)\n    pred = prediction_frame(val, horizon, "lstm", rates, "val")\n    metrics = metrics_table(pred)\n    pred.to_csv(out / "previsoes_validacao.csv.gz", index=False)\n    metrics.to_csv(out / "metricas_validacao.csv", index=False)\n    save_json(out / "completed.json", {\n        "fingerprint": m["fingerprint"], "seed": seed,\n        "epochs_ran": len(history), "best_val_mse_scaled": best,\n        "train_seconds": elapsed, "val_inference_seconds": inference_s,\n        "weights_sha256": file_sha(out / "best_weights.pt"),\n    })\n    return metrics\n\n\ndef predict_horizon(run_dir, horizon, arr):\n    import torch\n    out = Path(run_dir) / f"lstm_h{horizon}"\n    cfg = read_json(out / "config.json")\n    m = load_run(run_dir)\n    if cfg["fingerprint"] != m["fingerprint"] or cfg["horizon_min"] != horizon:\n        raise ValueError("Configuração incompatível com os dados.")\n    if (out / "completed.json").exists():\n        expected = read_json(out / "completed.json")["weights_sha256"]\n        if file_sha(out / "best_weights.pt") != expected:\n            raise ValueError("Pesos alterados depois do treino.")\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    model = make_model(cfg["hidden"])\n    model.load_state_dict(torch.load(out / "best_weights.pt",\n                                    map_location="cpu", weights_only=True))\n    model.to(device).eval()\n    s = cfg["scales"]\n    x = (arr["X"] - s["x_mean"]) / s["x_std"]\n    started = time.perf_counter()\n    chunks = []\n    with torch.no_grad():\n        for start in range(0, len(x), 2048):\n            xb = torch.from_numpy(x[start:start+2048].astype(np.float32)).to(device)\n            chunks.append(model(xb).cpu().numpy())\n    rate = np.concatenate(chunks) * s["y_std"] + s["y_mean"]\n    return rate, time.perf_counter() - started\n\n\ndef validation_results(run_dir):\n    run_dir = Path(run_dir)\n    baseline_validation(run_dir)\n    paths = [run_dir / "baselines" / "previsoes_validacao.csv.gz"]\n    for h in HORIZONS:\n        if not (run_dir / f"lstm_h{h}" / "completed.json").exists():\n            raise FileNotFoundError(f"Conclua o notebook da LSTM de {h} minutos.")\n        paths.append(run_dir / f"lstm_h{h}" / "previsoes_validacao.csv.gz")\n    predictions = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)\n    expected = set(map(tuple, predictions[predictions.model == "persistencia"]\n                       [["subject", "origin_ns", "horizon_min"]].to_numpy()))\n    for model in MODELS:\n        actual = set(map(tuple, predictions[predictions.model == model]\n                         [["subject", "origin_ns", "horizon_min"]].to_numpy()))\n        if actual != expected:\n            raise ValueError("Modelos com origens diferentes na validação.")\n    return predictions, metrics_table(predictions)\n\n\ndef select_horizon(run_dir, val_metrics, exploratory=False):\n    """Regra operacional fixa; nunca usa métricas de teste."""\n    run_dir = Path(run_dir)\n    if set(val_metrics.split) != {"val"}:\n        raise ValueError("Seleção recebe somente validação.")\n    summaries = []\n    for h in HORIZONS:\n        m = val_metrics[val_metrics.horizon_min == h]\n        macro = m[(m.scope == "macro") & (m.subset == "all")].set_index("model")\n        best = macro.loc[[x for x in MODELS if x != "lstm"], "mae_rate"].idxmin()\n        lstm, base = float(macro.loc["lstm", "mae_rate"]), float(macro.loc[best, "mae_rate"])\n        per = m[(m.scope == "subject") & (m.subset == "all")].pivot(\n            index="subject", columns="model", values="mae_rate")\n        wins = float((per.lstm < per[best]).mean())\n        rapid = m[(m.scope == "macro") & (m.subset == "rapid")].set_index("model")\n        rapid_lstm, rapid_base = float(rapid.loc["lstm", "mae_rate"]), float(rapid.loc[best, "mae_rate"])\n        opposite_lstm = float(rapid.loc["lstm", "sign_opposite"])\n        opposite_base = float(rapid.loc[best, "sign_opposite"])\n        correct_lstm = float(rapid.loc["lstm", "sign_correct"])\n        correct_base = float(rapid.loc[best, "sign_correct"])\n        # Sem suporte em mudanças rápidas, a candidatura fica inconclusiva.\n        candidate = bool(base > 0 and lstm < base and wins > .5 and\n                         np.isfinite(rapid_lstm) and rapid_lstm <= rapid_base and\n                         correct_lstm >= correct_base)\n        summaries.append({\n            "horizon_min": h, "best_baseline": best, "mae_lstm": lstm,\n            "mae_best_baseline": base, "gain": 1-lstm/base if base > 0 else None,\n            "fraction_subjects_improved": wins, "rapid_mae_lstm": rapid_lstm,\n            "rapid_mae_baseline": rapid_base,\n            "rapid_sign_opposite_lstm": opposite_lstm,\n            "rapid_sign_opposite_baseline": opposite_base, "candidate": candidate,\n            "rapid_sign_correct_lstm": correct_lstm,\n            "rapid_sign_correct_baseline": correct_base,\n        })\n    candidates = [s for s in summaries if s["candidate"]]\n    chosen = None\n    if candidates:\n        max_gain = max(s["gain"] for s in candidates)\n        # Empate operacional: diferença de até 0,02 no ganho relativo.\n        near = [s for s in candidates if max_gain - s["gain"] <= .02]\n        chosen = max(s["horizon_min"] for s in near)\n    selection = {\n        "fingerprint": load_run(run_dir)["fingerprint"],\n        "chosen_horizon_min": chosen, "validation_summary": summaries,\n        "exploratory_prior_test_access": bool(exploratory),\n        "rule": "Ganho macro >0, maioria dos participantes, sem piora de MAE "\n                "ou acerto de sinal em mudanças rápidas. Empate <=0.02 favorece 15. "\n                "Prever zero conta como erro direcional quando a taxa real é não nula.",\n        "validation_sha256": hashlib.sha256(\n            val_metrics.to_csv(index=False).encode()).hexdigest(),\n        "model_hashes": {str(h): file_sha(run_dir / f"lstm_h{h}" / "best_weights.pt")\n                         for h in HORIZONS},\n    }\n    path = run_dir / "selecao_horizonte.json"\n    selection = clean_json(selection)\n    if path.exists():\n        if read_json(path) != selection:\n            raise ValueError("Seleção já congelada e diferente. Use novo RUN_NAME.")\n    else:\n        save_json(path, selection)\n    return selection\n\n\ndef test_results(run_dir):\n    run_dir = Path(run_dir)\n    if not (run_dir / "selecao_horizonte.json").exists():\n        raise ValueError("Congele a seleção na validação antes do teste.")\n    selection = read_json(run_dir / "selecao_horizonte.json")\n    if selection["fingerprint"] != load_run(run_dir)["fingerprint"]:\n        raise ValueError("Seleção e dados incompatíveis.")\n    for h in HORIZONS:\n        if selection["model_hashes"][str(h)] != file_sha(\n                run_dir / f"lstm_h{h}" / "best_weights.pt"):\n            raise ValueError("Modelo mudou depois da escolha do horizonte.")\n    arr = load_split(run_dir, "test")\n    frames, timings = [], {}\n    for h in HORIZONS:\n        rates = baseline_rates(arr["X"])\n        rates["lstm"], elapsed = predict_horizon(run_dir, h, arr)\n        timings[str(h)] = elapsed\n        frames.extend(prediction_frame(arr, h, name, rate, "test")\n                      for name, rate in rates.items())\n    pred = pd.concat(frames, ignore_index=True)\n    metrics = metrics_table(pred)\n    out = run_dir / "final"\n    out.mkdir(exist_ok=True)\n    pred.to_csv(out / "previsoes_teste.csv.gz", index=False)\n    metrics.to_csv(out / "metricas_teste.csv", index=False)\n    save_json(out / "tempos_inferencia_teste.json", timings)\n    save_json(out / "test_done.json", {\n        "selection_sha256": file_sha(run_dir / "selecao_horizonte.json"),\n        "fingerprint": selection["fingerprint"],\n    })\n    return pred, metrics\n\n\ndef skill_table(metrics):\n    rows = []\n    for (split, h, subset, scope, sid), g in metrics.groupby(\n            ["split", "horizon_min", "subset", "scope", "subject"], dropna=False):\n        by = g.set_index("model")\n        if "lstm" not in by.index:\n            continue\n        for b in MODELS[:-1]:\n            if b not in by.index:\n                continue\n            base, value = float(by.loc[b, "mae_rate"]), float(by.loc["lstm", "mae_rate"])\n            rows.append(dict(split=split, horizon_min=h, subset=subset, scope=scope,\n                             subject=sid, baseline=b,\n                             gain=1-value/base if base > 0 and np.isfinite(value) else np.nan))\n    return pd.DataFrame(rows)\n\n\ndef plot_history(run_dir, horizon):\n    import matplotlib.pyplot as plt\n    out = Path(run_dir) / f"lstm_h{horizon}"\n    df = pd.read_csv(out / "historico.csv")\n    fig, ax = plt.subplots(figsize=(8, 3))\n    ax.plot(df.epoch, df.train_mse_scaled, marker="o", label="Treino")\n    ax.plot(df.epoch, df.val_mse_scaled, marker="o", label="Validação")\n    ax.set(xlabel="Época", ylabel="MSE padronizado", title=f"LSTM — {horizon} minutos")\n    ax.legend(); fig.tight_layout()\n    fig.savefig(out / "aprendizado.png", dpi=140)\n    return fig\n\n\ndef plot_results(predictions, run_dir):\n    import matplotlib.pyplot as plt\n    from matplotlib.patches import FancyArrowPatch\n    out = Path(run_dir) / "final"\n    out.mkdir(exist_ok=True)\n    figs = []\n    d = predictions[predictions.model == "lstm"].copy()\n    d["origin"] = pd.to_datetime(d.origin)\n    fig, axes = plt.subplots(1, 2, figsize=(11, 4))\n    for ax, h in zip(axes, HORIZONS):\n        part = d[d.horizon_min == h]\n        step = max(1, len(part) // 5000)\n        ax.scatter(part.rate_true.iloc[::step], part.rate_pred.iloc[::step],\n                   s=6, alpha=.25)\n        lo = min(part.rate_true.min(), part.rate_pred.min())\n        hi = max(part.rate_true.max(), part.rate_pred.max())\n        ax.plot([lo, hi], [lo, hi], "--", color="black")\n        ax.set(title=f"+{h} min", xlabel="Taxa real (mg/dL/min)",\n               ylabel="Taxa prevista (mg/dL/min)")\n    fig.tight_layout(); fig.savefig(out / "taxa_real_vs_prevista.png", dpi=150)\n    figs.append(fig)\n    chosen = None\n    for sid, group in d[d.horizon_min == 5].groupby("subject", sort=True):\n        group = group.sort_values("origin")\n        segments = group.origin.diff().ne(pd.Timedelta(minutes=5)).cumsum()\n        for _, segment in group.groupby(segments):\n            if len(segment) >= 25:\n                chosen = (sid, segment.origin.iloc[0], segment.origin.iloc[24])\n                break\n        if chosen:\n            break\n    if chosen:\n        sid, start, end = chosen\n        panel = d[(d.subject == sid) & d.origin.between(start, end)]\n        fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)\n        g = panel[panel.horizon_min == 5].sort_values("origin")\n        axes[0].plot(g.origin, g.glucose_current, color="black")\n        axes[0].set(ylabel="CGM (mg/dL)", title=f"Primeira janela elegível — Subject {sid}")\n        for ax, h in zip(axes[1:], HORIZONS):\n            part = panel[panel.horizon_min == h].sort_values("origin")\n            ax.plot(part.origin, part.rate_true, label="Taxa futura real")\n            ax.plot(part.origin, part.rate_pred, label="LSTM")\n            ax.axhline(0, color="gray", lw=.7)\n            ax.set(ylabel=f"+{h} min\\nmg/dL/min"); ax.legend()\n        axes[-1].set_xlabel("Origem da previsão (taxas alinhadas à origem)")\n        fig.tight_layout(); fig.savefig(out / "comparacao_temporal.png", dpi=150)\n        figs.append(fig)\n    examples = d.sort_values(["subject", "origin", "horizon_min"]).head(6)\n    fig, axes = plt.subplots(2, 3, figsize=(11, 6))\n    for ax, (_, row) in zip(axes.flat, examples.iterrows()):\n        ax.add_patch(FancyArrowPatch((-.4*row.vector_x, -.4*row.vector_y),\n                                    (.6*row.vector_x, .6*row.vector_y),\n                                    arrowstyle="-|>", mutation_scale=25, lw=2))\n        ax.set(xlim=(-1, 1), ylim=(-1, 1), aspect="equal",\n               title=f"{row.rate_pred:+.2f} mg/dL/min | +{int(row.horizon_min)} min\\n"\n                     f"Real: {row.rate_true:+.2f} | Subject {int(row.subject)}")\n        ax.axis("off")\n    fig.suptitle("Primeiras previsões do teste — escala visual fixa")\n    fig.tight_layout(); fig.savefig(out / "setas_continuas.png", dpi=150)\n    figs.append(fig)\n    # Primeiro episódio rápido de cada horizonte, sem escolher o melhor acerto.\n    fig, axes = plt.subplots(2, 1, figsize=(12, 6))\n    for ax, h in zip(axes, HORIZONS):\n        part = d[d.horizon_min == h].sort_values(["subject", "origin"])\n        rapid = part[part.rate_true.abs() >= 1]\n        if rapid.empty:\n            ax.text(.5, .5, "Sem episódio rápido elegível", ha="center",\n                    transform=ax.transAxes)\n            continue\n        first = rapid.iloc[0]\n        window = part[(part.subject == first.subject) &\n                      part.origin.between(first.origin-pd.Timedelta(minutes=30),\n                                          first.origin+pd.Timedelta(minutes=90))]\n        ax.plot(window.origin, window.rate_true, label="Taxa futura real")\n        ax.plot(window.origin, window.rate_pred, label="LSTM")\n        ax.axvline(first.origin, color="gray", ls="--")\n        ax.set(title=f"Primeiro episódio rápido — +{h} min, Subject {int(first.subject)}",\n               ylabel="mg/dL/min"); ax.legend()\n    fig.tight_layout(); fig.savefig(out / "primeiro_episodio_rapido.png", dpi=150)\n    figs.append(fig)\n    return figs\n\n\ndef write_report(run_dir, val_metrics, test_metrics):\n    run_dir = Path(run_dir)\n    selection, manifest = read_json(run_dir / "selecao_horizonte.json"), load_run(run_dir)\n    all_metrics = pd.concat([val_metrics, test_metrics], ignore_index=True)\n    all_metrics.to_csv(run_dir / "final" / "metricas.csv", index=False)\n    skills = skill_table(all_metrics)\n    skills.to_csv(run_dir / "final" / "ganhos.csv", index=False)\n    rows = all_metrics[(all_metrics.scope == "macro") & (all_metrics.subset == "all")]\n    lines = [\n        "# AZT1D — Seta contínua: resultados executados",\n        "",\n        f"- Execução: {run_dir.name}",\n        f"- Modo rápido: {manifest[\'config\'][\'quick\']}",\n        f"- Participantes: {manifest[\'participants\']}",\n        f"- Horizonte escolhido antes do teste: {selection[\'chosen_horizon_min\']}",\n        f"- Teste já consultado em trabalhos anteriores: "\n        f"{selection[\'exploratory_prior_test_access\']}",\n        "- Taxa média futura em mg/dL/min; sem classes e sem simulação de doses.",\n        "",\n        "| Partição | Horizonte | Modelo | MAE taxa (média por pessoa) | RMSE glicemia (média por pessoa) |",\n        "|---|---:|---|---:|---:|",\n    ]\n    for r in rows.itertuples():\n        lines.append(f"| {r.split} | {r.horizon_min} | {r.model} | "\n                     f"{r.mae_rate:.4f} | {r.rmse_glucose:.3f} |")\n    lines += ["", "## Interpretação", "",\n              "A seleção usa validação, ganho sobre baseline e comportamento nas mudanças rápidas.",\n              "Consulte selecao_horizonte.json para a regra e a decisão congelada.",\n              "Se o horizonte escolhido for null, nenhum apresentou evidência suficiente pela regra operacional.",\n              "As métricas de teste confirmam ou contradizem a decisão; não reajuste os modelos neste teste.",\n              "", "## Limitações", "",\n              "Janelas sobrepostas não são amostras independentes. Avaliação temporal nas mesmas pessoas.",\n              "Agrupamento em 5 minutos; diferenças dos timestamps reais estão na auditoria.",\n              "Bins conflitantes e lacunas foram descartados, podendo mudar a população avaliada.",\n              "O modo rápido verifica execução; não sustenta conclusão sobre toda a base.",\n              "A seta resume a mudança média futura do CGM e não detecta todos os picos intermediários.",\n              "A escala angular é uma escolha visual do projeto, sem validação clínica.",\n              "", "## Fonte", "",\n              "Khamesian et al. AZT1D, versão 1. https://doi.org/10.17632/gk9m674wcx.1",\n              ""]\n    (run_dir / "final" / "resumo.md").write_text("\\n".join(lines), encoding="utf-8")\n    return run_dir / "final" / "resumo.md"\n\n\ndef export_results(run_dir):\n    run_dir = Path(run_dir)\n    zip_path = run_dir.parent / f"{run_dir.name}_resultados.zip"\n    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:\n        z.write(__file__, arcname=str(Path(run_dir.name) / "codigo" / "azt1d_core.py"))\n        for p in sorted(run_dir.rglob("*")):\n            if p.is_file() and "prepared" not in p.relative_to(run_dir).parts:\n                z.write(p, arcname=str(Path(run_dir.name) / p.relative_to(run_dir)))\n    return zip_path\n\n'
if CORE_PATH.exists() and CORE_PATH.read_text(encoding="utf-8") != CORE_SOURCE:
    raise RuntimeError("Código existente diferente. Escolha outro PROJECT_PATH.")
CORE_PATH.write_bytes(CORE_SOURCE.encode("utf-8"))
print("Código comum salvo:", CORE_PATH)

In [ ]:
# @title Carregar código compartilhado deste experimento
import hashlib, importlib.util, sys
EXPECTED_CORE_SHA = "d20343826d24973c0ffd22afeb2966612e613e61fa4df7d56c7d2269acb23f63"
if not CORE_PATH.exists():
    raise FileNotFoundError("Execute primeiro a célula que salva o código compartilhado.")
if hashlib.sha256(CORE_PATH.read_bytes()).hexdigest() != EXPECTED_CORE_SHA:
    raise RuntimeError("Código comum diferente desta versão. Use nova pasta de projeto.")
spec = importlib.util.spec_from_file_location("azt1d_core", CORE_PATH)
core = importlib.util.module_from_spec(spec)
sys.modules["azt1d_core"] = core
spec.loader.exec_module(core)

## 3. Fonte de dados e configuração

Escolha **upload_zip** para enviar o ZIP do computador, **drive_zip** para um ZIP
já no Drive ou **drive_folder** para CSVs extraídos.

Horários são mantidos sem conversão de fuso. DAYFIRST=False interpreta datas
ambíguas como mês/dia; confira as amostras na auditoria. Se sua cópia usa dia/mês,
configure True antes de preparar. Datas ISO não dependem dessa escolha.

O alias **Readings (CGM / BGM)** é excluído por padrão. Só habilite após confirmar
que as leituras usadas são CGM. Valores 40 e 400 não são excluídos automaticamente.

In [ ]:
# @title Configuração da base
DATA_SOURCE = "upload_zip" # @param ["upload_zip", "drive_zip", "drive_folder"]
DRIVE_ZIP = "/content/drive/MyDrive/AZT1D.zip" # @param {type:"string"}
DRIVE_DATA_DIR = "/content/drive/MyDrive/AZT1D 2025/CGM Records" # @param {type:"string"}
QUICK = True # @param {type:"boolean"}
if "AZT1D_QUICK" in os.environ:
    QUICK = os.environ["AZT1D_QUICK"].lower() in ("1", "true", "yes")
DAYFIRST = False # @param {type:"boolean"}
ALLOW_MIXED_ALIAS = False # @param {type:"boolean"}
SEED = 42
DATE_FORMAT = "mixed"  # ou formato exato, como "%Y-%m-%d %H:%M:%S"

if os.environ.get("AZT1D_LOCAL_DATA"):
    DATA_DIR = Path(os.environ["AZT1D_LOCAL_DATA"])
elif not USE_DRIVE and (REPO_ROOT / "AZT1D 2025" / "CGM Records").is_dir():
    DATA_DIR = REPO_ROOT / "AZT1D 2025" / "CGM Records"
elif not USE_DRIVE:
    raise FileNotFoundError("Defina AZT1D_LOCAL_DATA apontando para a pasta CGM Records.")
elif DATA_SOURCE == "drive_folder":
    DATA_DIR = Path(DRIVE_DATA_DIR)
else:
    if DATA_SOURCE == "upload_zip":
        from google.colab import files
        print("Selecione somente o arquivo ZIP do AZT1D.")
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
        if len(zip_names) != 1:
            raise ValueError("Envie exatamente um ZIP.")
        archive = Path(zip_names[0])
        del uploaded
    else:
        archive = Path(DRIVE_ZIP)
    if not archive.is_file():
        raise FileNotFoundError(archive)
    input_parent = Path("/content") if Path("/content").exists() else PROJECT_DIR
    DATA_DIR = input_parent / ("azt1d_raw_" + core.file_sha(archive)[:12])
    marker = DATA_DIR / ".extraction_complete"
    if not marker.exists():
        core.safe_extract_zip(archive, DATA_DIR)
        marker.write_text("ok")
print("Dados:", DATA_DIR)
print("Modo:", "rápido" if QUICK else "completo")

In [ ]:
# @title 4. Preparar dados, alvos e divisões temporais
manifest = core.prepare_dataset(
    DATA_DIR, RUN_DIR, quick=QUICK, dayfirst=DAYFIRST,
    date_format=DATE_FORMAT, allow_mixed_alias=ALLOW_MIXED_ALIAS, seed=SEED)
print("Participantes incluídos:", manifest["participants"])
print("Janelas:", manifest["counts"])
print("Tempo de preparação (s):", round(manifest["preparation_seconds"], 2))
print("Exclusões registradas:", manifest["excluded"])

## 5. Auditar antes de treinar

A política remove duplicatas completas, fecha intervalos de 5 min à direita,
invalida conflitos e não interpola. Entrada e alvos ficam na mesma partição:
60% treino, 20% validação, 20% teste por pessoa.

As janelas têm 12 entradas e três passos futuros disponíveis. Os horizontes
usam as mesmas origens. No modo rápido: três primeiros participantes elegíveis
e até 20.000 janelas de treino.

Confira datas, exclusões, descartes e intervalos reais. Se detectar data ou unidade
incorreta, corrija a configuração e use outro RUN_NAME.

In [ ]:
import pandas as pd
from IPython.display import display
audit = pd.read_csv(RUN_DIR / "auditoria_participantes.csv")
display(audit)
display(pd.read_csv(RUN_DIR / "auditoria_arquivos.csv"))
display(pd.read_csv(RUN_DIR / "divisoes_temporais.csv"))
intervals = pd.read_csv(RUN_DIR / "auditoria_horizontes_reais.csv")
display(intervals.groupby("horizon_min")[
    ["raw_spacing_min", "raw_spacing_median", "raw_spacing_max",
     "raw_spacing_p95_abs_deviation"]].agg(["min", "median", "max"]))

In [ ]:
# @title Exemplo de treino, sem acessar o teste
import matplotlib.pyplot as plt
import numpy as np
train = core.load_split(RUN_DIR, "train")
i = 0
display(pd.DataFrame({
    "horario_entrada": pd.to_datetime(train["input_start_ns"][i] +
                                     np.arange(12) * core.STEP_NS),
    "glicemia": train["X"][i, :, 0],
}))
print("Origem:", pd.Timestamp(train["origin_ns"][i]))
print("Alvos reais de treino [5, 15] min:", train["future"][i])
print("Taxas reais de treino [5, 15] mg/dL/min:", train["y"][i])
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(np.arange(-55, 1, 5), train["X"][i, :, 0], "o-", label="Entrada")
ax.scatter([5, 15], train["future"][i], color="orange", label="Alvos de treino")
ax.axvline(0, color="gray", ls="--")
ax.set(xlabel="Minutos em relação à origem", ylabel="CGM (mg/dL)")
ax.legend(); plt.show()

## 6. Baselines na validação

Taxa zero, última diferença de 5 min e inclinação da reta ajustada às quatro
leituras mais recentes. A taxa real é $(G(t+h)-G(t))/h$.

**macro** é a média das métricas por pessoa; **pooled** usa todas as janelas juntas.
Mudanças rápidas ($|r|\geq1$) são um subconjunto de diagnóstico, não classes.

In [ ]:
baseline_metrics = core.baseline_validation(RUN_DIR)
display(baseline_metrics.query("scope == 'macro'")[
    ["horizon_min", "model", "subset", "n", "mae_rate", "rmse_rate",
     "mae_glucose", "sign_correct"]])
print("Próximo passo: treinar os dois horizontes neste notebook.")

## Modelo LSTM: horizonte de 5 minutos

Previsão da taxa média `(G(t+5)-G(t))/5`, usando 12 leituras anteriores.


In [ ]:
# @title 2. Conferir execução preparada
HORIZON = 5
manifest = core.load_run(RUN_DIR)
print("Participantes:", manifest["participants"])
print("Modo rápido:", manifest["config"]["quick"])
print("Épocas máximas:", 3 if manifest["config"]["quick"] else 15)
print("Janelas:", manifest["counts"])
import torch
print("Dispositivo:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# @title 3. Treinar e salvar melhor estado de validação
metrics = core.train_horizon(RUN_DIR, HORIZON, seed=manifest["config"]["seed"])

## 4. Resultados de validação

Pesos são salvos a cada melhora. Se a sessão for interrompida, os melhores pesos
permanecem no Drive. Rodar novamente um treino incompleto recomeça do início,
com a mesma semente; não retoma o otimizador. Treinos concluídos são reutilizados.

Depois de congelar a seleção no seção de avaliação final, novos treinos exigem outro RUN_NAME.

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
display(metrics.query("scope == 'macro'")[
    ["horizon_min", "model", "subset", "n", "mae_rate", "rmse_rate",
     "mae_glucose", "rmse_glucose", "sign_correct", "sign_opposite"]])
fig = core.plot_history(RUN_DIR, HORIZON)
plt.show()
print("Artefatos:", RUN_DIR / f"lstm_h{HORIZON}")

In [ ]:
# @title 5. Primeiras previsões contínuas de validação
import pandas as pd
preview = pd.read_csv(RUN_DIR / f"lstm_h{HORIZON}" / "previsoes_validacao.csv.gz")
display(preview.head(10)[
    ["subject", "origin", "target_time", "rate_true", "rate_pred",
     "glucose_pred", "angle_deg", "vector_x", "vector_y"]])
print("Cada valor produz uma inclinação contínua, sem três classes.")

## Modelo LSTM: horizonte de 15 minutos

Previsão da taxa média `(G(t+15)-G(t))/15`, com as mesmas origens temporais.


In [ ]:
# @title 2. Conferir execução preparada
HORIZON = 15
manifest = core.load_run(RUN_DIR)
print("Participantes:", manifest["participants"])
print("Modo rápido:", manifest["config"]["quick"])
print("Épocas máximas:", 3 if manifest["config"]["quick"] else 15)
print("Janelas:", manifest["counts"])
import torch
print("Dispositivo:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# @title 3. Treinar e salvar melhor estado de validação
metrics = core.train_horizon(RUN_DIR, HORIZON, seed=manifest["config"]["seed"])

## 4. Resultados de validação

Pesos são salvos a cada melhora. Se a sessão for interrompida, os melhores pesos
permanecem no Drive. Rodar novamente um treino incompleto recomeça do início,
com a mesma semente; não retoma o otimizador. Treinos concluídos são reutilizados.

Depois de congelar a seleção no seção de avaliação final, novos treinos exigem outro RUN_NAME.

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
display(metrics.query("scope == 'macro'")[
    ["horizon_min", "model", "subset", "n", "mae_rate", "rmse_rate",
     "mae_glucose", "rmse_glucose", "sign_correct", "sign_opposite"]])
fig = core.plot_history(RUN_DIR, HORIZON)
plt.show()
print("Artefatos:", RUN_DIR / f"lstm_h{HORIZON}")

In [ ]:
# @title 5. Primeiras previsões contínuas de validação
import pandas as pd
preview = pd.read_csv(RUN_DIR / f"lstm_h{HORIZON}" / "previsoes_validacao.csv.gz")
display(preview.head(10)[
    ["subject", "origin", "target_time", "rate_true", "rate_pred",
     "glucose_pred", "angle_deg", "vector_x", "vector_y"]])
print("Cada valor produz uma inclinação contínua, sem três classes.")

## Comparação dos horizontes e avaliação da seta

A escolha é congelada com as métricas de validação antes de calcular as métricas
do teste. Em seguida, o notebook gera previsões, visualizações e relatório.


In [ ]:
# @title 2. Comparar somente validação
import pandas as pd
from IPython.display import display
val_predictions, val_metrics = core.validation_results(RUN_DIR)
display(val_metrics.query("scope == 'macro'")[
    ["horizon_min", "model", "subset", "n", "mae_rate", "rmse_rate",
     "mae_glucose", "sign_correct", "sign_opposite"]])
display(core.skill_table(val_metrics).query("scope == 'macro'"))

## 3. Congelar escolha antes do teste

Regra operacional reproduzível:

1. Melhor baseline por horizonte: média do MAE de taxa por pessoa.
2. Exigir ganho da LSTM nessa média e em mais da metade das pessoas.
3. Exigir que não piore MAE nem acerto de sinal nas mudanças rápidas.
4. Maior ganho entre candidatos vence; diferença até **0,02 no ganho relativo**
é empate operacional que favorece 15 min pela antecedência.
5. Sem candidato, salvar **null** e reportar ausência de evidência suficiente.

O empate é uma convenção, não um critério clínico. Previsão zero conta como erro
de direção quando a taxa real é não nula, para não favorecer a persistência
por nunca apontar sinal oposto. Sinais opostos continuam no relatório.

Se sua equipe já usou o teste para decisões anteriores, marque
**TESTE_JA_CONSULTADO=True**: a avaliação será identificada como exploratória.

In [ ]:
# @title Congelar seleção
TESTE_JA_CONSULTADO = True # @param {type:"boolean"}
if "AZT1D_TESTE_JA_CONSULTADO" in os.environ:
    TESTE_JA_CONSULTADO = os.environ["AZT1D_TESTE_JA_CONSULTADO"].lower() in ("1", "true", "yes")
selection = core.select_horizon(RUN_DIR, val_metrics, exploratory=TESTE_JA_CONSULTADO)
display(pd.DataFrame(selection["validation_summary"]))
print("Horizonte escolhido:", selection["chosen_horizon_min"])
print("Decisão salva antes de calcular métricas no teste.")

## 4. Avaliação final

A partir daqui você terá visto o teste. Não mude a configuração para melhorar
seus números. O modo rápido verifica o fluxo; não conclui o estudo do AZT1D.

In [ ]:
test_predictions, test_metrics = core.test_results(RUN_DIR)
display(test_metrics.query("scope == 'macro'")[
    ["horizon_min", "model", "subset", "n", "mae_rate", "rmse_rate",
     "mae_glucose", "rmse_glucose", "sign_correct", "sign_opposite"]])
display(core.skill_table(test_metrics).query("scope == 'macro'"))

## 5. Visualizar taxa e seta

$\theta=\arctan(r/r_{\rm ref})$; vetor unitário $(1,q)/\sqrt{1+q^2}$;
$q=r/r_{\rm ref}$; $r_{\rm ref}=1$ mg/dL/min. Sem arredondamento em classes.

A escala é visual e fixa. A taxa é a mudança média até +5 ou +15 min, não a
derivada instantânea no final nem a garantia de trajetória monotônica.

In [ ]:
import matplotlib.pyplot as plt
figures = core.plot_results(test_predictions, RUN_DIR)
plt.show()

In [ ]:
# @title 6. Gerar relatório e ZIP
report = core.write_report(RUN_DIR, val_metrics, test_metrics)
from IPython.display import Markdown
display(Markdown(report.read_text(encoding="utf-8")))
RESULT_ZIP = core.export_results(RUN_DIR)
print("ZIP com métricas, previsões, gráficos, código e pesos:", RESULT_ZIP)
print("Caches de dados permanecem na pasta do projeto e não são duplicados no ZIP.")

In [ ]:
# @title 7. Baixar resultados (opcional)
DOWNLOAD_RESULTS = False # @param {type:"boolean"}
if DOWNLOAD_RESULTS:
    if USE_DRIVE:
        from google.colab import files
        files.download(str(RESULT_ZIP))
    else:
        print("Resultados disponíveis em:", RESULT_ZIP.resolve())


## Referências e interpretação

- Dataset: Khamesian et al., **AZT1D v1**, [DOI 10.17632/gk9m674wcx.1](https://doi.org/10.17632/gk9m674wcx.1), CC BY 4.0.
- Horizontes de 5 e 15 min: [Meriem et al., ICAEE 2024](https://doi.org/10.1109/ICAEE61760.2024.10783291).
- Baselines e variabilidade: [Mosquera-Lopez e Jacobs, 2022](https://doi.org/10.1177/19322968211042621).
- Coerência dinâmica: [De Bois et al., ICONIP 2019](https://doi.org/10.1007/978-3-030-36718-3_43).
- Tendência e futuro: [Freckmann et al., 2018](https://doi.org/10.1089/dia.2018.0112).

A configuração de 32 unidades, as janelas e a escala visual são decisões do
protótipo. O experimento prevê **CGM futuro**, usa as mesmas pessoas em períodos
distintos e não constitui validação clínica.